# Modelo MLP

## 1. Librerias

In [ ]:
import os
import tensorflow as tf
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import s3fs
from sklearn.impute   import SimpleImputer
from sklearn.metrics  import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from google.colab import userdata


In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "eu-west-1"

S3_BASE = "s3://tfm-mu-bd/processed"
S3_OUT  = "s3://tfm-mu-bd/outputs/mlp_baseline"
s3      = s3fs.S3FileSystem()

EXCLUDE_COLS = [
    "record_id", "dx", "labels", "categories",
    "is_adverse", "snomed_unknown", "dx_multi_hot", "cat_arrhythmia",
    "cat_axis_deviation", "cat_conduction_block", "cat_interval_change", "cat_ischemia",
    "cat_morphology_change", "cat_repolarization",
    "cat_sinus_rhythm", "cat_structural"
]

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## 3. Carga y preparación

In [ ]:
def prepare_features(df, feature_cols):
    X = df[feature_cols].copy()
    for col in feature_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.values.astype(np.float32)


def load_and_prepare():
    print("Cargando datos desde S3...")
    df_train = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_train.parquet")
    df_val   = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_val.parquet")
    df_test  = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_test.parquet")

    feature_cols = [c for c in df_train.columns if c not in EXCLUDE_COLS]

        imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(prepare_features(df_train, feature_cols))
    X_val   = imputer.transform(prepare_features(df_val,   feature_cols))
    X_test  = imputer.transform(prepare_features(df_test,  feature_cols))

        scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    y_train = df_train["is_adverse"].astype(int).values
    y_val   = df_val["is_adverse"].astype(int).values
    y_test  = df_test["is_adverse"].astype(int).values

        X_trainval = np.vstack([X_train, X_val])
    y_trainval = np.concatenate([y_train, y_val])

    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)
    class_weight = {0: (neg + pos) / (2 * neg),
                    1: (neg + pos) / (2 * pos)}
    print(f"  class_weight: {class_weight}")
    print(f"  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
    print(f"  Train+Val: {X_trainval.shape} | Features: {len(feature_cols)}")

    return (X_train, y_train, X_val, y_val,
            X_test, y_test, X_trainval, y_trainval,
            feature_cols, class_weight)

## 3. Modelo y evaluación

In [ ]:
def build_mlp(input_dim):
    inp = keras.Input(shape=(input_dim,), name="features")

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)

    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs=inp, outputs=out, name="MLP_baseline")
    return model

In [ ]:
def train_mlp(X_train, y_train, X_val, y_val, class_weight, input_dim):

    print("\n BÚSQUEDA DE HIPERPARÁMETROS")

    best_val_auc = 0
    best_config  = None
    search_results = []


    for lr in [1e-3, 5e-4, 1e-4]:
        for batch_size in [64, 128, 256]:
            print(f"\n  lr={lr} | batch={batch_size}")
            tf.random.set_seed(RANDOM_STATE)

            model = build_mlp(input_dim)
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss="binary_crossentropy",
                metrics=[keras.metrics.AUC(name="auc")]
            )

            cb = [
                callbacks.EarlyStopping(
                    monitor="val_auc", patience=10,
                    restore_best_weights=True, mode="max"
                )
            ]

            hist = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=50,
                batch_size=batch_size,
                class_weight=class_weight,
                callbacks=cb,
                verbose=0
            )

            val_auc = max(hist.history["val_auc"])
            print(f"    Val AUC: {val_auc:.4f} (epoch {np.argmax(hist.history['val_auc'])+1})")
            search_results.append({
                "lr": lr, "batch_size": batch_size, "val_auc": val_auc
            })

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_config  = {"lr": lr, "batch_size": batch_size}

            del model
            gc.collect()

    print(f"\nMejor config: {best_config} | Val AUC: {best_val_auc:.4f}")
    return best_config, search_results


def train_final(best_config, X_trainval, y_trainval, X_val, y_val,
                class_weight, input_dim):

    print("\n ENTRENAMIENTO FINAL (train+val = 85%)")
    tf.random.set_seed(RANDOM_STATE)

    model = build_mlp(input_dim)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=best_config["lr"]),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )

    cb = [
        callbacks.EarlyStopping(
            monitor="val_auc", patience=15,
            restore_best_weights=True, mode="max"
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_auc", factor=0.5, patience=7,
            mode="max", min_lr=1e-6, verbose=1
        )
    ]

    history = model.fit(
        X_trainval, y_trainval,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=best_config["batch_size"],
        class_weight=class_weight,
        callbacks=cb,
        verbose=1
    )

    model.summary()
    return model, history


In [ ]:
def evaluate_full(model, X, y, split_name, threshold=0.5):
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    auc  = roc_auc_score(y, y_prob)
    f1   = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    ap   = average_precision_score(y, y_prob)
    cm   = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)

    print(f"\n  [{split_name}]")
    print(f"    AUC-ROC:     {auc:.4f}")
    print(f"    AUC-PR:      {ap:.4f}")
    print(f"    F1:          {f1:.4f}")
    print(f"    Precision:   {prec:.4f}")
    print(f"    Recall:      {rec:.4f}")
    print(f"    Specificity: {specificity:.4f}")
    print(f"    Confusion matrix:\n{cm}")

    return {
        "split": split_name,
        "auc_roc": round(auc,  4),
        "auc_pr": round(ap,   4),
        "f1": round(f1,   4),
        "precision": round(prec, 4),
        "recall": round(rec,  4),
        "specificity": round(specificity, 4),
        "cm": cm.tolist(),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
    }


## 4. Visualización

In [ ]:
def plot_learning_curve(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["loss"],     label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Curva de pérdida — MLP")
    axes[0].legend()

    axes[1].plot(history.history["auc"],     label="Train AUC")
    axes[1].plot(history.history["val_auc"], label="Val AUC")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("AUC-ROC")
    axes[1].set_title("Curva de AUC — MLP")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("./mlp_learning_curve.png", dpi=150)
    plt.close()
    print("  mlp_learning_curve.png guardado")


def plot_roc_pr(model, X_test, y_test):
    y_prob = model.predict(X_test, verbose=0).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f"AUC = {auc:.4f}", color="purple")
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("Curva ROC — MLP (Test)")
    axes[0].legend()

    prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec_c, prec_c, label=f"AUC-PR = {ap:.4f}", color="darkorange")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Curva Precision-Recall — MLP (Test)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("./mlp_roc_pr.png", dpi=150)
    plt.close()
    print("  mlp_roc_pr.png guardado")

def plot_confusion_matrix(cm):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
                xticklabels=["No adverso", "Adverso"],
                yticklabels=["No adverso", "Adverso"])
    plt.title("MLP — Matriz de confusión (Test)")
    plt.ylabel("Real")
    plt.xlabel("Predicho")
    plt.tight_layout()
    plt.savefig("./mlp_confusion_matrix.png", dpi=150)
    plt.close()
    print("  mlp_confusion_matrix.png guardado")


def plot_search_results(search_results):
    df_s = pd.DataFrame(search_results)
    pivot = df_s.pivot(index="batch_size", columns="lr", values="val_auc")

    plt.figure(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd")
    plt.title("Val AUC — Grid de hiperparámetros MLP")
    plt.tight_layout()
    plt.savefig("./mlp_hyperparam_search.png", dpi=150)
    plt.close()
    print("  mlp_hyperparam_search.png guardado")

## 5. Subir los datos

In [ ]:
def upload_to_s3():
    files = [
        "./mlp_learning_curve.png",
        "./mlp_roc_pr.png",
        "./mlp_confusion_matrix.png",
        "./mlp_hyperparam_search.png",
        "./mlp_results.json",
    ]
    for local in files:
        remote = f"{S3_OUT}/{local.lstrip('./')}"
        try:
            s3.put(local, remote)
            print(f"{local} → {remote}")
        except Exception as e:
            print(f"ERROR {local}: {e}")


## 6. Ejecución principal

In [ ]:
if __name__ == "__main__":

    (X_train, y_train, X_val, y_val,
     X_test, y_test, X_trainval, y_trainval,
     feature_cols, class_weight) = load_and_prepare()

    input_dim = X_train.shape[1]

    best_config, search_results = train_mlp(
        X_train, y_train, X_val, y_val, class_weight, input_dim
    )

    final_model, history = train_final(
        best_config, X_trainval, y_trainval,
        X_val, y_val, class_weight, input_dim
    )

    print("\n EVALUACIÓN FINAL (TEST)")
    r_test = evaluate_full(final_model, X_test, y_test, "test")

    print("\nGenerando visualizaciones")
    plot_learning_curve(history)
    plot_roc_pr(final_model, X_test, y_test)
    plot_confusion_matrix(np.array(r_test["cm"]))
    plot_search_results(search_results)

    results = {
        "best_config": best_config,
        "search_results": search_results,
        "architecture": "256-128-64-32-1",
        "test": r_test
    }
    with open("./mlp_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("\n  mlp_results.json guardado")

    print("\nSubiendo artefactos a S3")
    upload_to_s3()

    print("\nMLP BASELINE COMPLETO ")
    print(f"\nRESUMEN TEST:")
    print(f" AUC-ROC:{r_test['auc_roc']}")
    print(f" AUC-PR: {r_test['auc_pr']}")
    print(f" F1: {r_test['f1']}")
    print(f" Recall: {r_test['recall']}")
    print(f" Specificity: {r_test['specificity']}")